# Schedule building with `ScheduleBuilder` (reference)

**Deep dive:** fluent configuration of payment/roll schedules — frequency, stubs, CDS IMM mode, end-of-month rolls, and `ScheduleErrorPolicy`.


## Concept

- **`ScheduleBuilder(start, end)`** accumulates **mutable** state: set **`frequency`**, optional **`stub_rule`**, optional **`adjust_with`**, optional **`end_of_month`**, optional **`cds_imm()` / `imm()`**, optional **`error_policy`**, then **`build()`**.
- **Stubs** control where an irregular first/last period sits: **short** vs **long**, **front** vs **back**.
- **End-of-month**: when the start is month-end, rolls track month-ends (February, holidays, etc.).
- **CDS IMM** / **IMM** modes align rolls to market-standard IMM patterns.
- **`ScheduleErrorPolicy`** chooses between **strict** failure, **warnings** when a calendar id is missing, or **graceful empty** schedules.


## API walkthrough

### Basics: `frequency` and `build()`

Frequency accepts **`Tenor.parse`-compatible strings** (e.g. `"6M"`, `"3M"`) or a `Tenor` object. Default stub is **none** — periods divide evenly when possible.


In [1]:
from datetime import date

try:
    from finstack.core.dates import ScheduleBuilder, StubKind, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    print("StubKind.NONE:", StubKind.NONE)
    b = ScheduleBuilder(date(2024, 1, 15), date(2025, 1, 15))
    print("Fresh builder:", b)
    b.frequency("6M")
    b.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
    sched = b.build()
    print("len:", len(sched))
    print("dates:", [str(d) for d in sched.dates])


StubKind.NONE: none
Fresh builder: ScheduleBuilder(start=2024-01-15, end=2025-01-15, freq=3M)
len: 3
dates: ['2024-01-16', '2024-07-15', '2025-01-15']


### Stub rules (`StubKind`)

**`NONE`**, **`SHORT_FRONT`**, **`SHORT_BACK`**, **`LONG_FRONT`**, **`LONG_BACK`** — use `from_name` for string parsing.


In [2]:
from datetime import date

try:
    from finstack.core.dates import ScheduleBuilder, StubKind, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    # End deliberately off a clean 3M grid so off-grid stub behavior is visible.
    start, end = date(2024, 3, 20), date(2027, 5, 20)
    for stub in [StubKind.NONE, StubKind.SHORT_BACK, StubKind.SHORT_FRONT, StubKind.LONG_BACK, StubKind.LONG_FRONT]:
        b = ScheduleBuilder(start, end)
        b.frequency("3M")
        b.stub_rule(stub)
        b.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "target2")
        try:
            s = b.build()
        except ValueError as exc:
            print(f"{stub!s:16s} error={exc}")
            continue
        print(f"{stub!s:16s} n={len(s.dates):2d} first={s.dates[1]} last={s.dates[-1]}")
    print("from_name('short_front') == SHORT_FRONT:", StubKind.from_name("short_front") == StubKind.SHORT_FRONT)

none             n=14 first=2024-06-20 last=2027-05-20
short_back       n=14 first=2024-06-20 last=2027-05-20
short_front      n=14 first=2024-05-20 last=2027-05-20
long_back        n=13 first=2024-06-20 last=2027-05-20
long_front       n=14 first=2024-05-20 last=2027-05-20
from_name('short_front') == SHORT_FRONT: True


### CDS IMM dates and generic IMM rolls

- **`cds_imm()`** — standard CDS roll pattern (quarterly IMM-style anchors).
- **`imm()`** — futures-style IMM schedule (distinct from plain quarterly).


In [3]:
from datetime import date

try:
    from finstack.core.dates import ScheduleBuilder, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    b_cds = ScheduleBuilder(date(2024, 3, 20), date(2029, 3, 20))
    b_cds.cds_imm()
    b_cds.adjust_with(BusinessDayConvention.FOLLOWING, "usny")
    cds = b_cds.build()
    print("CDS IMM (first 6):", [str(d) for d in cds.dates[:6]])

    b_imm = ScheduleBuilder(date(2024, 1, 15), date(2025, 1, 15))
    b_imm.imm()
    b_imm.adjust_with(BusinessDayConvention.FOLLOWING, "usny")
    imm = b_imm.build()
    print("IMM mode dates:", [str(d) for d in imm.dates])


CDS IMM (first 6): ['2024-03-20', '2024-06-20', '2024-09-20', '2024-12-20', '2025-03-20', '2025-06-20']
IMM mode dates: ['2024-03-20', '2024-06-20', '2024-09-18', '2024-12-18']


### End-of-month convention

Enable with **`end_of_month(True)`** so monthly rolls respect month-end alignment (e.g. Jan-31 → Feb-29 in leap years).


In [4]:
from datetime import date

try:
    from finstack.core.dates import ScheduleBuilder, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    b = ScheduleBuilder(date(2024, 1, 31), date(2025, 1, 31))
    b.frequency("1M")
    b.end_of_month(True)
    b.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
    s = b.build()
    print("EOM monthly (sample):", [str(d) for d in s.dates[:4]], "...", [str(d) for d in s.dates[-3:]])


EOM monthly (sample): ['2024-01-31', '2024-02-29', '2024-03-29', '2024-04-30'] ... ['2024-11-29', '2024-12-31', '2025-01-31']


### `ScheduleErrorPolicy`

- **`STRICT`** — missing calendar or invalid spec **raises**.
- **`MISSING_CALENDAR_WARNING`** — warn and **skip** adjustment if the calendar id is unknown.
- **`GRACEFUL_EMPTY`** — return an **empty** `Schedule` and record warnings / fallback flags.


In [5]:
from datetime import date

try:
    from finstack.core.dates import ScheduleBuilder, ScheduleErrorPolicy, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    bad = "definitely_not_a_calendar"

    b = ScheduleBuilder(date(2024, 1, 15), date(2025, 1, 15))
    b.frequency("6M")
    b.adjust_with(BusinessDayConvention.FOLLOWING, bad)
    b.error_policy(ScheduleErrorPolicy.GRACEFUL_EMPTY)
    s = b.build()
    print("GRACEFUL_EMPTY len:", len(s), "used_graceful_fallback:", s.used_graceful_fallback())
    print("warnings:", s.warnings)

    b2 = ScheduleBuilder(date(2024, 1, 15), date(2025, 1, 15))
    b2.frequency("6M")
    b2.adjust_with(BusinessDayConvention.FOLLOWING, bad)
    b2.error_policy(ScheduleErrorPolicy.MISSING_CALENDAR_WARNING)
    s2 = b2.build()
    print("MISSING_CALENDAR_WARNING has_warnings:", s2.has_warnings())
    print("first warning:", s2.warnings[0] if s2.warnings else "(none)")


GRACEFUL_EMPTY len: 0 used_graceful_fallback: True
warnings: ["graceful fallback triggered: Calendar not found: 'definitely_not_a_calendar'"]
MISSING_CALENDAR_WARNING has_warnings: True
first warning: calendar id 'definitely_not_a_calendar' not found; adjustment skipped


## Practical example

Three real-world patterns:

1. **USD semi-annual corporate-style bond** — 6M frequency, **Modified Following** on **USNY**.
2. **EUR quarterly swap** — 3M, **short back stub**, **TARGET2**.
3. **CDS standard rolls** — `cds_imm()` with **USNY** adjustment.


In [6]:
from datetime import date

try:
    from finstack.core.dates import ScheduleBuilder, StubKind, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    issue, maturity = date(2024, 1, 15), date(2029, 1, 15)
    b_bond = ScheduleBuilder(issue, maturity)
    b_bond.frequency("6M")
    b_bond.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
    bond = b_bond.build()
    print("1) Semi-annual bond (USNY):", len(bond.dates), "dates")
    print("   head:", [str(d) for d in bond.dates[:3]])
    print("   tail:", [str(d) for d in bond.dates[-2:]])

    b_swap = ScheduleBuilder(date(2024, 3, 20), date(2027, 3, 20))
    b_swap.frequency("3M")
    b_swap.stub_rule(StubKind.SHORT_BACK)
    b_swap.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, "target2")
    swap = b_swap.build()
    print("2) Quarterly swap (T2, short back stub):", len(swap.dates), "dates")
    print("   rolls:", [str(d) for d in swap.dates[1:5]], "...")

    b_cds = ScheduleBuilder(date(2024, 3, 20), date(2029, 3, 20))
    b_cds.cds_imm()
    b_cds.adjust_with(BusinessDayConvention.FOLLOWING, "usny")
    cds = b_cds.build()
    print("3) CDS IMM (USNY):", len(cds.dates), "dates")
    print("   first rolls:", [str(d) for d in cds.dates[:5]])


1) Semi-annual bond (USNY): 11 dates
   head: ['2024-01-16', '2024-07-15', '2025-01-15']
   tail: ['2028-07-17', '2029-01-16']
2) Quarterly swap (T2, short back stub): 13 dates
   rolls: ['2024-06-20', '2024-09-20', '2024-12-20', '2025-03-20'] ...
3) CDS IMM (USNY): 21 dates
   first rolls: ['2024-03-20', '2024-06-20', '2024-09-20', '2024-12-20', '2025-03-20']


## Takeaways

- Configure **`ScheduleBuilder`** with **imperative** calls (not a fluent returning `self`); then **`build()`** once.
- Choose **`stub_rule`** when the tenor does not divide the interval evenly.
- Use **`cds_imm()`** vs plain **`frequency("3M")`** when you need **CDS market roll alignment**; use **`imm()`** for **futures IMM**-style schedules.
- **`end_of_month(True)`** is essential for **bonds** that pay on month-end when the issue is on month-end.
- Pick **`ScheduleErrorPolicy`** for **production** robustness: strict in tests, warnings or graceful in exploratory notebooks.
